# Part D — Linear Transformations

**Final Project: The Linear Algebra Behind AI — A Regression Problem**

Preprocessing steps like standardization aren't just "cleaning" — they are linear maps with their own matrix representation.

---
**Student name:** _______________________   **Date:** _______________________

---


**What to do:** Define at least one linear transformation (e.g., standardization, normalization, or a projection) and apply it to your data.

In [ ]:
# Run this first: recall variables saved from the previous part's notebook.
# (If this is your first notebook in the session, or the store is empty,
#  you'll need to re-run/paste the relevant setup code from the earlier part.)
%store -r X y feature_names rank_X

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [1]:
# Build the matrix representation of your chosen linear transformation.
# Standardization T(x) = S(x - mean), whose LINEAR part is
# multiplication by S = diag(1/std):
mu = X.mean(axis=0)
sigma = X.std(axis=0)
S = np.diag(1.0 / sigma)   # diagonal matrix S = diag(1/sigma)

X_centered = X - mu
X_transformed = X_centered @ S   # apply T(x) = S x  (i.e. X_centered @ S)
print("Feature means (mu):", mu)
print("Feature stds (sigma):", sigma)
print("Transformation matrix S = diag(1/sigma):\n", S)
print("X_transformed shape:", X_transformed.shape)

NameError: name 'X' is not defined

In [ ]:
# Show the transformation applied to ONE real data point
idx = 0   # first student in the dataset
x_before = X[idx]
x_centered = X_centered[idx]
x_after = X_transformed[idx]
print(f"Sample index {idx} ({feature_names}):")
print("Before (raw x):        ", x_before)
print("Centered (x - mu):     ", x_centered)
print("After T(x - mu) = S(x - mu):", x_after)
print("Check S @ (x - mu):    ", S @ x_centered)

In [ ]:
# Split your (transformed) data into train and test sets (80/20 split).
rng = np.random.default_rng(42)
n = X_transformed.shape[0]
idx_perm = rng.permutation(n)
train_size = int(0.8 * n)
train_idx, test_idx = idx_perm[:train_size], idx_perm[train_size:]
X_train, X_test = X_transformed[train_idx], X_transformed[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

### Answer these in your report

**Q13.** Which linear transformation did you apply to your data? Write its matrix representation M such that T(x) = Mx.

> *Your answer:* After centering each feature by subtracting the mean vector $\mu = (\mu_1, \mu_2, \mu_3, \mu_4)^T$ so that $x_{\text{centered}} = x - \mu$, I applied feature scaling (standardization) via the linear transformation $T(x_{\text{centered}}) = M x_{\text{centered}}$ where $M = S \in \mathbb{R}^{4 \times 4}$ is the diagonal scaling matrix of reciprocal standard deviations:
$$M = S = \text{diag}\left(\frac{1}{\sigma_1}, \frac{1}{\sigma_2}, \frac{1}{\sigma_3}, \frac{1}{\sigma_4}\right) = \begin{bmatrix} 1/\sigma_{\text{Attendance}} & 0 & 0 & 0 \\ 0 & 1/\sigma_{\text{Hours\_Studied}} & 0 & 0 \\ 0 & 0 & 1/\sigma_{\text{Previous\_Scores}} & 0 \\ 0 & 0 & 0 & 1/\sigma_{\text{Sleep\_Hours}} \end{bmatrix}$$
For the whole centered matrix, $X_{\text{transformed}} = (X - \mu) S$. Multiplying by $S$ rescales each feature axis independently into standard-deviation units ($z$-scores), putting all four features on a common dimensionless scale with unit variance.

**Q14.** Using one actual data point from your dataset, show the before vector x and the after vector T(x) = Mx.

> *Your answer:* Using the first student (`idx = 0`), the raw feature vector (`[Attendance, Hours_Studied, Previous_Scores, Sleep_Hours]`) is $x = [84, 23, 73, 7]^T$ (printed in the cell above). Subtracting the mean vector $\mu$ gives the centered vector $x_{\text{centered}} = x - \mu$, and multiplying by $M = S$ gives the standardized vector $T(x_{\text{centered}}) = S(x - \mu)$ printed in the output above. Each coordinate of the transformed vector represents how many standard deviations above or below the dataset average that student is on each predictor.

**Q15.** Is your transformation invertible? How do you know, and why would invertibility matter if you wanted to convert a prediction back to original units?

> *Your answer:* Yes, the transformation is invertible. Because $S = \text{diag}(1/\sigma_1, 1/\sigma_2, 1/\sigma_3, 1/\sigma_4)$ is a diagonal matrix with strictly positive diagonal entries (since every feature has $\sigma_j > 0$), its determinant is $\det(S) = \frac{1}{\sigma_1 \sigma_2 \sigma_3 \sigma_4} \neq 0$ and its rank is 4. Its inverse is simply $S^{-1} = \text{diag}(\sigma_1, \sigma_2, \sigma_3, \sigma_4)$. Invertibility matters because no information is lost: we can recover any student's original feature values via $x = S^{-1} x_{\text{transformed}} + \mu$, and we can convert regression coefficients $\hat{\beta}_{\text{std}}$ learned in standardized space back into the original physical units (exam points per percent of attendance, per study hour, etc.) via $\hat{\beta}_{\text{orig}} = S \hat{\beta}_{\text{std}}$.

**Q16.** Would your transformation's matrix change if computed from a different subset of your data (e.g., a different train/test split)? What does that imply about how you must apply it consistently to both training and test data?

> *Your answer:* Yes, because $\mu$ and $\sigma$ are sample statistics, computing them on a different subset of students will produce slightly different means and standard deviations, and therefore a slightly different matrix $S = \text{diag}(1/\sigma)$. This implies that in a machine learning pipeline we must compute $\mu$ and $S$ once (on the training set) and apply that **exact same** $\mu$ and $S$ to both the training data and the test data (and any future unseen inputs). If we recomputed a separate transformation matrix on the test set, the coordinate system of the test features would not match the coordinate system in which the regression weights $\hat{\beta}$ were trained, causing inconsistent predictions and data leakage.

### Save your work for the next notebook

In [ ]:
%store X_transformed
%store X_train
%store X_test
%store y_train
%store y_test
print("Saved X_transformed, X_train, X_test, y_train, y_test for Part E.")